# Phase 3: Data Preparation and Transformation

This notebook prepares the validated MovieLens datasets for recommendation model development.

Activities:
- Load validated datasets
- Clean and prepare data
- Perform chronological train/test split
- Transform interaction data
- Save prepared datasets in CSV and Parquet formats

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

PROJECT_DIR = "/content/drive/MyDrive/recommendation-pipeline"

RAW_DATA_DIR = os.path.join(PROJECT_DIR, "data", "raw")
PROCESSED_DATA_DIR = os.path.join(PROJECT_DIR, "data", "processed")
REPORT_DIR = os.path.join(PROJECT_DIR, "reports")

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print("Project directories configured successfully!")

Mounted at /content/drive
Project directories configured successfully!


In [2]:
#Load the validated datasets
ratings_df = pd.read_parquet(
    os.path.join(PROCESSED_DATA_DIR, "ratings.parquet")
)

movies_df = pd.read_parquet(
    os.path.join(PROCESSED_DATA_DIR, "movies.parquet")
)

users_df = pd.read_parquet(
    os.path.join(PROCESSED_DATA_DIR, "users.parquet")
)

print("Datasets loaded successfully!")

print("Ratings:", ratings_df.shape)
print("Movies:", movies_df.shape)
print("Users:", users_df.shape)

display(ratings_df.head())

Datasets loaded successfully!
Ratings: (100836, 4)
Movies: (9742, 3)
Users: (610, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## Data Cleaning

In [3]:
# Check missing values
print("Missing values before cleaning:")

print("Ratings:", ratings_df.isnull().sum().sum())
print("Movies:", movies_df.isnull().sum().sum())
print("Users:", users_df.isnull().sum().sum())

# Check duplicate records
print("\nDuplicate records before cleaning:")

print("Ratings:", ratings_df.duplicated().sum())
print("Movies:", movies_df.duplicated().sum())
print("Users:", users_df.duplicated().sum())

# Remove exact duplicate records
ratings_df = ratings_df.drop_duplicates()
movies_df = movies_df.drop_duplicates()
users_df = users_df.drop_duplicates()

# Remove records with missing essential values
ratings_df = ratings_df.dropna(
    subset=["userId", "movieId", "rating", "timestamp"]
)

movies_df = movies_df.dropna(
    subset=["movieId", "title", "genres"]
)

users_df = users_df.dropna(
    subset=["userId", "age", "occupation", "location"]
)

# Keep only valid rating values
ratings_df = ratings_df[
    ratings_df["rating"].between(0.5, 5.0)
].copy()

print("\nCleaning completed!")

print("Ratings after cleaning:", ratings_df.shape)
print("Movies after cleaning:", movies_df.shape)
print("Users after cleaning:", users_df.shape)

Missing values before cleaning:
Ratings: 0
Movies: 0
Users: 0

Duplicate records before cleaning:
Ratings: 0
Movies: 0
Users: 0

Cleaning completed!
Ratings after cleaning: (100836, 4)
Movies after cleaning: (9742, 3)
Users after cleaning: (610, 4)


## Time-Based Train/Test Split

In [4]:
# Sort ratings chronologically
ratings_df = ratings_df.sort_values(
    by="timestamp"
).reset_index(drop=True)

# Calculate 80% training boundary
split_index = int(len(ratings_df) * 0.8)

# Split into training and testing datasets
train_df = ratings_df.iloc[:split_index].copy()

test_df = ratings_df.iloc[split_index:].copy()

print("Time-based split completed!")

print("Training records:", len(train_df))
print("Testing records:", len(test_df))

print("\nTraining date range:")
print(
    pd.to_datetime(train_df["timestamp"], unit="s").min(),
    "to",
    pd.to_datetime(train_df["timestamp"], unit="s").max()
)

print("\nTesting date range:")
print(
    pd.to_datetime(test_df["timestamp"], unit="s").min(),
    "to",
    pd.to_datetime(test_df["timestamp"], unit="s").max()
)

Time-based split completed!
Training records: 80668
Testing records: 20168

Training date range:
1996-03-29 18:36:55 to 2016-03-22 08:26:02

Testing date range:
2016-03-22 08:26:11 to 2018-09-24 14:27:30


## Data Transformation and Storage

In [5]:
# Convert Unix timestamps into readable dates
for dataframe in [train_df, test_df]:
    dataframe["datetime"] = pd.to_datetime(
        dataframe["timestamp"],
        unit="s"
    )

    dataframe["year"] = dataframe["datetime"].dt.year
    dataframe["month"] = dataframe["datetime"].dt.month
    dataframe["day"] = dataframe["datetime"].dt.day

# Add movie information
train_df = train_df.merge(
    movies_df,
    on="movieId",
    how="left"
)

test_df = test_df.merge(
    movies_df,
    on="movieId",
    how="left"
)

# Add user information
train_df = train_df.merge(
    users_df,
    on="userId",
    how="left"
)

test_df = test_df.merge(
    users_df,
    on="userId",
    how="left"
)

print("Data transformation completed!")

display(train_df.head())

Data transformation completed!


,userId,movieId,rating,timestamp,datetime,year,month,day,title,genres,age,occupation,location
0,429,165,4.0,828124615,1996-03-29 18:36:55,1996,3,29,Die Hard: With a Vengeance (1995),Action|Crime|Thriller,34,Doctor,Vancouver
1,429,595,5.0,828124615,1996-03-29 18:36:55,1996,3,29,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX,34,Doctor,Vancouver
2,429,434,4.0,828124615,1996-03-29 18:36:55,1996,3,29,Cliffhanger (1993),Action|Adventure|Thriller,34,Doctor,Vancouver
3,429,590,5.0,828124615,1996-03-29 18:36:55,1996,3,29,Dances with Wolves (1990),Adventure|Drama|Western,34,Doctor,Vancouver
4,429,588,5.0,828124615,1996-03-29 18:36:55,1996,3,29,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,34,Doctor,Vancouver


In [6]:
# Save training data
train_df.to_csv(
    os.path.join(PROCESSED_DATA_DIR, "train_data.csv"),
    index=False
)

train_df.to_parquet(
    os.path.join(PROCESSED_DATA_DIR, "train_data.parquet"),
    index=False
)

# Save testing data
test_df.to_csv(
    os.path.join(PROCESSED_DATA_DIR, "test_data.csv"),
    index=False
)

test_df.to_parquet(
    os.path.join(PROCESSED_DATA_DIR, "test_data.parquet"),
    index=False
)

print("Training and testing datasets saved successfully!")

Training and testing datasets saved successfully!
